In [5]:
import boto3
import pandas as pd
import sagemaker
import os

In [33]:
from sagemaker.core.helper.session_helper import Session, get_execution_role

session = Session()
role = get_execution_role()
# bucket = session.default_bucket()
bucket = "bkt-deloitte-aug26-hyd"
print("Bucket:", bucket)
print("Role:", role)

Bucket: bkt-deloitte-aug26-hyd
Role: arn:aws:iam::781843541123:role/service-role/AmazonSageMaker-ExecutionRole-20260827T100868


In [7]:
print(os.getcwd())
print(os.listdir())

/home/sagemaker-user
['.bashrc', '.agent', '.kiro', '.claude', '.sagemaker_sql_editor_api_cache', '.local', '.ipython', '.aws', '.npm', '.jupyter', '.ipynb_checkpoints', '.cache', '.config', '.virtual_documents', 'Usecase_Testing.ipynb']


## Data Preparation:

In [8]:
df = pd.read_csv(f"s3://{bucket}/raw-data/FreshMart_Customer_Membership.csv")
df.head()

,Customer_ID,Customer_Name,Age,Gender,Income,Orders_Last_Year,Average_Order_Value,City,App_Usage_Hours,Membership_Years,Premium_Member
0,C000001,Aanya Verma,43,Male,97822.0,13,3086.0,Surat,5.3,3,0
1,C000002,Neha Kumar,22,Male,92126.0,13,3929.0,Hyderabad,3.9,10,0
2,C000003,Vivaan Das,55,Female,86821.0,11,2598.0,Kolkata,4.4,2,0
3,C000004,Meera Kulkarni,53,Male,95948.0,14,3810.0,Indore,5.6,1,1
4,C000005,Nisha Nair,49,Female,74764.0,3,5011.0,Hyderabad,3.9,12,0


In [9]:
df.shape

(50500, 11)

In [10]:
df.isnull().sum()

Customer_ID               0
Customer_Name             0
Age                       0
Gender                    0
Income                 1006
Orders_Last_Year          0
Average_Order_Value    1000
City                   1010
App_Usage_Hours        1012
Membership_Years          0
Premium_Member            0
dtype: int64

In [11]:
df.duplicated().sum()

499

In [12]:
df = df.drop_duplicates()

In [13]:
df.duplicated().sum()

0

In [14]:
df = df.dropna(subset=["Premium_Member"])
X = df.drop(
    columns=[
        "Premium_Member",
        "Customer_ID",
        "Customer_Name"
    ]
)
y = df["Premium_Member"]

In [16]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing y:", y.isna().sum())
print(y.value_counts())

X shape: (50001, 8)
y shape: (50001,)
Missing y: 0
Premium_Member
0    37500
1    12501
Name: count, dtype: int64


## Train/Test Split

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

## Handle Missing Values + Encoding

In [18]:
cat_cols = X.select_dtypes("object").columns
num_cols = X.select_dtypes(exclude="object").columns

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        num_cols
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        cat_cols
    )
])

In [20]:
print("y_train nulls:", y_train.isna().sum())
print("y_test nulls:", y_test.isna().sum())

print(y_train.value_counts(dropna=False))

y_train nulls: 0
y_test nulls: 0
Premium_Member
0    30013
1     9987
Name: count, dtype: int64


## Model Experiments

1) Model V1 → Logistic Regression

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

lr = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

2) Random Forest

In [21]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

# Compare Models

In [24]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def metrics(y, pred):
    return [
        accuracy_score(y, pred),
        precision_score(y, pred, zero_division=0),
        recall_score(y, pred, zero_division=0),
        f1_score(y, pred, zero_division=0)
    ]

results = pd.DataFrame([
    ["Logistic Regression", "V1", *metrics(y_test, lr_pred)],
    ["Random Forest", "V2", *metrics(y_test, rf_pred)]
], columns=[
    "Model",
    "Version",
    "Accuracy",
    "Precision",
    "Recall",
    "F1"
])
results

,Model,Version,Accuracy,Precision,Recall,F1
0,Logistic Regression,V1,0.747325,0.367347,0.007160,0.014046
1,Random Forest,V2,0.740826,0.443149,0.120923,0.190000


# Select Best Model

In [25]:
best_model_name = results.loc[
    results["F1"].idxmax(),
    "Model"
]
print("Candidate Model:", best_model_name)

Candidate Model: Random Forest


In [26]:
candidate = rf if best_model_name == "Random Forest" else lr
candidate

,steps,"[('prep', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Save Experiment Results

In [27]:
results.to_csv(
    "experiment_results.csv",
    index=False
)

In [30]:
session.upload_data(
    path="experiment_results.csv",
    bucket=bucket,
    key_prefix="experiments"
)

's3://bkt-deloitte-aug26-hyd/experiments/experiment_results.csv'

# Save Models

In [31]:
import joblib

joblib.dump(lr, "model_v1.joblib")
joblib.dump(rf, "model_v2.joblib")

['model_v2.joblib']

In [32]:
session.upload_data(
    path="model_v1.joblib",
    bucket=bucket,
    key_prefix="models"
)

session.upload_data(
    path="model_v2.joblib",
    bucket=bucket,
    key_prefix="models"
)

's3://bkt-deloitte-aug26-hyd/models/model_v2.joblib'

In [38]:
bucket = "bkt-deloitte-aug26-hyd"
inputPath = "s3://bkt-deloitte-aug26-hyd/raw-data/FreshMart_Customer_Membership.csv"
print(bucket)
print(inputPath)

bkt-deloitte-aug26-hyd
s3://bkt-deloitte-aug26-hyd/raw-data/FreshMart_Customer_Membership.csv


In [39]:
CANDIDATE_MODEL = "Random Forest"

# Run the complete pipeline code:

In [45]:
# import sys
# import sagemaker

# print("Python:", sys.executable)
# print("SageMaker location:", sagemaker.__file__)
# print("SageMaker module:", sagemaker)

# try:
#     from sagemaker.mlops.workflow.pipeline import Pipeline
#     print("✅ SageMaker SDK v3 Pipeline import works")
# except Exception as e:
#     print("❌ Pipeline import failed:", e)

Python: /opt/conda/bin/python
SageMaker location: /opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py
SageMaker module: <module 'sagemaker' from '/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py'>
✅ SageMaker SDK v3 Pipeline import works


In [46]:
# from sagemaker.mlops.workflow.pipeline import Pipeline
# import inspect
# print(inspect.signature(Pipeline))

(name: str = '', parameters: Optional[Sequence[sagemaker.core.workflow.parameters.Parameter]] = None, pipeline_experiment_config: Optional[sagemaker.mlops.workflow.pipeline_experiment_config.PipelineExperimentConfig] = <sagemaker.mlops.workflow.pipeline_experiment_config.PipelineExperimentConfig object at 0x7f033ec656a0>, mlflow_config: Optional[sagemaker.core.shapes.shapes.MlflowConfig] = None, steps: Optional[Sequence[Union[sagemaker.mlops.workflow.steps.Step, sagemaker.core.workflow.step_outputs.StepOutput]]] = None, sagemaker_session: Optional[sagemaker.core.helper.session_helper.Session] = None, pipeline_definition_config: Optional[sagemaker.core.workflow.pipeline_definition_config.PipelineDefinitionConfig] = <sagemaker.core.workflow.pipeline_definition_config.PipelineDefinitionConfig object at 0x7f033eab05f0>)


In [47]:
# print(CANDIDATE_MODEL)

Random Forest


In [48]:
# import pandas as pd

# df = pd.read_csv(
#     "s3://bkt-deloitte-aug26-hyd/raw-data/FreshMart_Customer_Membership.csv"
# )
# print(df.columns.tolist())

['Customer_ID', 'Customer_Name', 'Age', 'Gender', 'Income', 'Orders_Last_Year', 'Average_Order_Value', 'City', 'App_Usage_Hours', 'Membership_Years', 'Premium_Member']


In [ ]:
# FreshMart CSV
#     ↓
# Preprocessing
#     ├─ remove duplicates
#     ├─ handle missing values
#     ├─ remove Customer_ID / Customer_Name
#     └─ 80:20 split, random_state=42
#     ↓
# Random Forest Training
#     ↓
# Evaluation
#     ├─ Accuracy
#     ├─ Precision
#     ├─ Recall
#     └─ F1
#     ↓
# Model Registration

In [50]:
# from sagemaker.core.shapes import ProcessingInput, ProcessingOutput, ProcessingS3Input

# import inspect

# print("ProcessingInput:")
# print(inspect.signature(ProcessingInput))

# print("\nProcessingS3Input:")
# print(inspect.signature(ProcessingS3Input))

# print("\nProcessingOutput:")
# print(inspect.signature(ProcessingOutput))

ProcessingInput:
(*, input_name: Union[str, sagemaker.core.helper.pipeline_variable.PipelineVariable], app_managed: Optional[bool] = Unassigned(), s3_input: Optional[sagemaker.core.shapes.shapes.ProcessingS3Input] = Unassigned(), dataset_definition: Optional[sagemaker.core.shapes.shapes.DatasetDefinition] = Unassigned()) -> None

ProcessingS3Input:
(*, s3_uri: Union[str, sagemaker.core.helper.pipeline_variable.PipelineVariable], s3_data_type: Union[str, sagemaker.core.helper.pipeline_variable.PipelineVariable], local_path: Union[str, sagemaker.core.helper.pipeline_variable.PipelineVariable, NoneType] = Unassigned(), s3_input_mode: Union[str, sagemaker.core.helper.pipeline_variable.PipelineVariable, NoneType] = Unassigned(), s3_data_distribution_type: Union[str, sagemaker.core.helper.pipeline_variable.PipelineVariable, NoneType] = Unassigned(), s3_compression_type: Union[str, sagemaker.core.helper.pipeline_variable.PipelineVariable, NoneType] = Unassigned()) -> None

ProcessingOutput:
(

In [51]:
# from sagemaker.train.model_trainer import ModelTrainer
# from sagemaker.train.configs import SourceCode, InputData, Compute
# from sagemaker.mlops.workflow.steps import ProcessingStep, TrainingStep
# from sagemaker.mlops.workflow.model_step import ModelStep

# import inspect

# print("ModelTrainer:")
# print(inspect.signature(ModelTrainer))

# print("\nSourceCode:")
# print(inspect.signature(SourceCode))

# print("\nInputData:")
# print(inspect.signature(InputData))

# print("\nCompute:")
# print(inspect.signature(Compute))

# print("\nProcessingStep:")
# print(inspect.signature(ProcessingStep))

# print("\nTrainingStep:")
# print(inspect.signature(TrainingStep))

# print("\nModelStep:")
# print(inspect.signature(ModelStep))

ModelTrainer:
(*, training_mode: sagemaker.train.model_trainer.Mode = <Mode.SAGEMAKER_TRAINING_JOB: 'SAGEMAKER_TRAINING_JOB'>, sagemaker_session: Optional[sagemaker.core.helper.session_helper.Session] = None, role: Optional[str] = None, base_job_name: Optional[str] = None, source_code: Optional[sagemaker.core.training.configs.SourceCode] = None, distributed: Optional[sagemaker.train.distributed.DistributedConfig] = None, compute: Optional[sagemaker.core.training.configs.Compute] = None, networking: Optional[sagemaker.core.training.configs.Networking] = None, stopping_condition: Optional[sagemaker.core.shapes.shapes.StoppingCondition] = None, training_image: Union[str, sagemaker.core.helper.pipeline_variable.PipelineVariable, NoneType] = None, training_image_config: Optional[sagemaker.core.shapes.shapes.TrainingImageConfig] = None, algorithm_name: Union[str, sagemaker.core.helper.pipeline_variable.PipelineVariable, NoneType] = None, output_data_config: Optional[sagemaker.core.shapes.sha

In [53]:
# ============================================================
# TASK 4: SAGEMAKER MLOPS PIPELINE - SDK v3
# ============================================================
#
# Flow:
# S3 Dataset
#      ↓
# Preprocessing
#      ↓
# Random Forest Training
#      ↓
# Evaluation
#      ↓
# Model Registration
#
# ============================================================

import os
import json
import boto3

# ------------------------------------------------------------
# SageMaker SDK v3 imports
# ------------------------------------------------------------

from sagemaker.core.helper.session_helper import (
    Session,
    get_execution_role
)

from sagemaker.core.workflow.pipeline_context import (
    PipelineSession
)

from sagemaker.core.workflow.properties import (
    PropertyFile
)

from sagemaker.core.processing import (
    ScriptProcessor
)

from sagemaker.core.shapes import (
    ProcessingInput,
    ProcessingOutput,
    ProcessingS3Input,
    ProcessingS3Output
)

from sagemaker.core import image_uris

from sagemaker.mlops.workflow.pipeline import (
    Pipeline
)

from sagemaker.mlops.workflow.steps import (
    ProcessingStep,
    TrainingStep
)

from sagemaker.mlops.workflow.model_step import (
    ModelStep
)

from sagemaker.train.model_trainer import (
    ModelTrainer
)

from sagemaker.core.training.configs import (
    SourceCode,
    InputData,
    Compute
)

from sagemaker.serve.model_builder import (
    ModelBuilder
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

BUCKET = "bkt-deloitte-aug26-hyd"

RAW_DATA = (
    f"s3://{BUCKET}/raw-data/"
    "FreshMart_Customer_Membership.csv"
)

PIPELINE_NAME = "FreshMart-ML-Pipeline"

MODEL_GROUP = "FreshMart-Customer-Membership"

TARGET = "Premium_Member"

REGION = boto3.Session().region_name

ROLE = get_execution_role()

pipeline_session = PipelineSession()


print("=" * 60)
print("FRESHMART SAGEMAKER MLOPS PIPELINE")
print("=" * 60)

print("Region       :", REGION)
print("Bucket       :", BUCKET)
print("Input        :", RAW_DATA)
print("Target       :", TARGET)
print("Candidate    : Random Forest")
print("Pipeline     :", PIPELINE_NAME)
print("Model Group  :", MODEL_GROUP)
print("Role         :", ROLE)


# ============================================================
# 2. PREPROCESSING SCRIPT
# ============================================================

preprocessing_code = r'''
import os
import pandas as pd

from sklearn.model_selection import train_test_split


INPUT_DIR = "/opt/ml/processing/input"

TRAIN_DIR = "/opt/ml/processing/train"

TEST_DIR = "/opt/ml/processing/test"


os.makedirs(TRAIN_DIR, exist_ok=True)

os.makedirs(TEST_DIR, exist_ok=True)


# ------------------------------------------------------------
# Read CSV
# ------------------------------------------------------------

files = [
    f for f in os.listdir(INPUT_DIR)
    if f.endswith(".csv")
]

input_file = os.path.join(
    INPUT_DIR,
    files[0]
)

df = pd.read_csv(input_file)

print("Original dataset shape:", df.shape)


# ------------------------------------------------------------
# Remove duplicate customer records
# ------------------------------------------------------------

df = df.drop_duplicates()

print(
    "After duplicate removal:",
    df.shape
)


# ------------------------------------------------------------
# Target column
# ------------------------------------------------------------

TARGET = "Premium_Member"


# ------------------------------------------------------------
# Remove rows where target is missing
# ------------------------------------------------------------

df = df.dropna(
    subset=[TARGET]
)


# ------------------------------------------------------------
# Handle missing values
# ------------------------------------------------------------

for col in df.columns:

    if col == TARGET:
        continue

    if df[col].dtype == "object":

        mode_value = df[col].mode()

        if len(mode_value) > 0:

            df[col] = df[col].fillna(
                mode_value.iloc[0]
            )

        else:

            df[col] = df[col].fillna(
                "Unknown"
            )

    else:

        df[col] = df[col].fillna(
            df[col].median()
        )


# ------------------------------------------------------------
# Remove Customer_ID and Customer_Name
# ------------------------------------------------------------

for col in [
    "Customer_ID",
    "Customer_Name"
]:

    if col in df.columns:

        df = df.drop(
            columns=[col]
        )


# ------------------------------------------------------------
# Convert Premium_Member to numeric
# ------------------------------------------------------------

if df[TARGET].dtype == "object":

    mapping = {

        "Yes": 1,

        "No": 0,

        "Y": 1,

        "N": 0,

        "True": 1,

        "False": 0,

        "Premium": 1,

        "Regular": 0

    }

    df[TARGET] = (
        df[TARGET]
        .astype(str)
        .str.strip()
        .map(mapping)
    )


df[TARGET] = pd.to_numeric(
    df[TARGET],
    errors="coerce"
)


df = df.dropna(
    subset=[TARGET]
)


df[TARGET] = df[TARGET].astype(int)


# ------------------------------------------------------------
# 80:20 Train/Test split
# ------------------------------------------------------------

train_df, test_df = train_test_split(

    df,

    test_size=0.20,

    random_state=42,

    stratify=df[TARGET]
)


# ------------------------------------------------------------
# Print required shapes
# ------------------------------------------------------------

print(
    "Cleaned dataset shape:",
    df.shape
)

print(
    "Training dataset shape:",
    train_df.shape
)

print(
    "Testing dataset shape:",
    test_df.shape
)


# ------------------------------------------------------------
# Save train.csv
# ------------------------------------------------------------

train_df.to_csv(

    os.path.join(
        TRAIN_DIR,
        "train.csv"
    ),

    index=False
)


# ------------------------------------------------------------
# Save test.csv
# ------------------------------------------------------------

test_df.to_csv(

    os.path.join(
        TEST_DIR,
        "test.csv"
    ),

    index=False
)


print("Preprocessing completed successfully.")
'''


with open(
    "preprocessing.py",
    "w"
) as f:

    f.write(
        preprocessing_code
    )


# ============================================================
# 3. TRAINING SCRIPT - RANDOM FOREST
# ============================================================

training_code = r'''
import os
import pickle
import pandas as pd

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

from sklearn.ensemble import (
    RandomForestClassifier
)


# ------------------------------------------------------------
# Training data location
# ------------------------------------------------------------

TRAIN_DIR = "/opt/ml/input/data/train"


files = [
    f for f in os.listdir(TRAIN_DIR)
    if f.endswith(".csv")
]


train_file = os.path.join(
    TRAIN_DIR,
    files[0]
)


df = pd.read_csv(
    train_file
)


# ------------------------------------------------------------
# Target
# ------------------------------------------------------------

TARGET = "Premium_Member"


X = df.drop(
    columns=[TARGET]
)

y = df[TARGET]


# ------------------------------------------------------------
# Identify feature types
# ------------------------------------------------------------

categorical_columns = (
    X.select_dtypes(
        include=["object"]
    ).columns.tolist()
)


numerical_columns = (
    X.select_dtypes(
        exclude=["object"]
    ).columns.tolist()
)


print(
    "Categorical columns:",
    categorical_columns
)

print(
    "Numerical columns:",
    numerical_columns
)


# ------------------------------------------------------------
# Numerical preprocessing
# ------------------------------------------------------------

numeric_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),

    (
        "scaler",
        StandardScaler()
    )
])


# ------------------------------------------------------------
# Categorical preprocessing
# ------------------------------------------------------------

categorical_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),

    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])


# ------------------------------------------------------------
# Column Transformer
# ------------------------------------------------------------

preprocessor = ColumnTransformer([

    (
        "num",
        numeric_pipeline,
        numerical_columns
    ),

    (
        "cat",
        categorical_pipeline,
        categorical_columns
    )
])


# ------------------------------------------------------------
# RANDOM FOREST
# Candidate selected from Task 3
# ------------------------------------------------------------

rf = RandomForestClassifier(

    n_estimators=100,

    random_state=42,

    n_jobs=-1
)


# ------------------------------------------------------------
# Complete ML Pipeline
# ------------------------------------------------------------

model = Pipeline([

    (
        "preprocessor",
        preprocessor
    ),

    (
        "model",
        rf
    )
])


# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

print("Training Random Forest...")

model.fit(
    X,
    y
)


print("Training completed.")


# ------------------------------------------------------------
# Save model
# ------------------------------------------------------------

model_dir = os.environ.get(
    "SM_MODEL_DIR",
    "/opt/ml/model"
)


os.makedirs(
    model_dir,
    exist_ok=True
)


model_path = os.path.join(
    model_dir,
    "model.pkl"
)


with open(
    model_path,
    "wb"
) as f:

    pickle.dump(
        model,
        f
    )


print(
    "Model saved:",
    model_path
)
'''


os.makedirs(
    "training_code",
    exist_ok=True
)


with open(
    "training_code/train.py",
    "w"
) as f:

    f.write(
        training_code
    )


# ============================================================
# 4. EVALUATION SCRIPT
# ============================================================

evaluation_code = r'''
import os
import json
import pickle
import tarfile

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


MODEL_DIR = (
    "/opt/ml/processing/model"
)

TEST_DIR = (
    "/opt/ml/processing/test"
)

OUTPUT_DIR = (
    "/opt/ml/processing/evaluation"
)


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# Find model artifact
# ------------------------------------------------------------

model_files = [
    f for f in os.listdir(MODEL_DIR)
    if f.endswith(".tar.gz")
]


model_tar = os.path.join(
    MODEL_DIR,
    model_files[0]
)


# ------------------------------------------------------------
# Extract model
# ------------------------------------------------------------

extract_dir = "/tmp/model"


os.makedirs(
    extract_dir,
    exist_ok=True
)


with tarfile.open(
    model_tar,
    "r:gz"
) as tar:

    tar.extractall(
        extract_dir
    )


# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

model_path = os.path.join(
    extract_dir,
    "model.pkl"
)


with open(
    model_path,
    "rb"
) as f:

    model = pickle.load(f)


# ------------------------------------------------------------
# Read test dataset
# ------------------------------------------------------------

test_files = [
    f for f in os.listdir(TEST_DIR)
    if f.endswith(".csv")
]


test_file = os.path.join(
    TEST_DIR,
    test_files[0]
)


test_df = pd.read_csv(
    test_file
)


TARGET = "Premium_Member"


X_test = test_df.drop(
    columns=[TARGET]
)

y_test = test_df[TARGET]


# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

predictions = model.predict(
    X_test
)


# ------------------------------------------------------------
# Calculate metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test,
    predictions
)


precision = precision_score(
    y_test,
    predictions,
    zero_division=0
)


recall = recall_score(
    y_test,
    predictions,
    zero_division=0
)


f1 = f1_score(
    y_test,
    predictions,
    zero_division=0
)


results = {

    "model": "Random Forest",

    "accuracy": float(
        accuracy
    ),

    "precision": float(
        precision
    ),

    "recall": float(
        recall
    ),

    "f1_score": float(
        f1
    )
}


print("=" * 50)

print("EVALUATION RESULTS")

print("=" * 50)

print(
    json.dumps(
        results,
        indent=4
    )
)


# ------------------------------------------------------------
# Save evaluation.json
# ------------------------------------------------------------

evaluation_file = os.path.join(
    OUTPUT_DIR,
    "evaluation.json"
)


with open(
    evaluation_file,
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=4
    )


print(
    "Evaluation saved:",
    evaluation_file
)
'''


with open(
    "evaluation.py",
    "w"
) as f:

    f.write(
        evaluation_code
    )


# ============================================================
# 5. SAGEMAKER SCIKIT-LEARN IMAGE
# ============================================================

SKLEARN_VERSION = "1.4-2"


sklearn_image = image_uris.retrieve(

    framework="sklearn",

    region=REGION,

    version=SKLEARN_VERSION,

    py_version="py3",

    instance_type="ml.m5.large",

    image_scope="training"
)


print("\nSageMaker SKLearn image:")

print(
    sklearn_image
)


# ============================================================
# 6. PREPROCESSING STEP
# ============================================================

processor = ScriptProcessor(

    role=ROLE,

    image_uri=sklearn_image,

    command=["python3"],

    instance_type="ml.m5.large",

    instance_count=1,

    sagemaker_session=pipeline_session
)


processing_args = processor.run(

    code="preprocessing.py",

    inputs=[

        ProcessingInput(

            input_name="raw-data",

            s3_input=ProcessingS3Input(

                s3_uri=RAW_DATA,

                s3_data_type="S3Prefix",

                local_path=
                    "/opt/ml/processing/input",

                s3_input_mode="File",

                s3_compression_type="None"
            )
        )
    ],

    outputs=[

        ProcessingOutput(

            output_name="train",

            s3_output=ProcessingS3Output(

                s3_uri=
                    f"s3://{BUCKET}/pipeline/train",

                local_path=
                    "/opt/ml/processing/train",

                s3_upload_mode="EndOfJob"
            )
        ),

        ProcessingOutput(

            output_name="test",

            s3_output=ProcessingS3Output(

                s3_uri=
                    f"s3://{BUCKET}/pipeline/test",

                local_path=
                    "/opt/ml/processing/test",

                s3_upload_mode="EndOfJob"
            )
        )
    ],

    wait=False
)


step_process = ProcessingStep(

    name="Preprocessing",

    step_args=processing_args
)


# ============================================================
# 7. TRAINING STEP
# ============================================================

trainer = ModelTrainer(

    training_image=sklearn_image,

    source_code=SourceCode(

        source_dir="training_code",

        entry_script="train.py"
    ),

    compute=Compute(

        instance_type="ml.m5.large",

        instance_count=1
    ),

    role=ROLE,

    base_job_name=
        "freshmart-random-forest",

    sagemaker_session=pipeline_session
)


train_args = trainer.train(

    input_data_config=[

        InputData(

            channel_name="train",

            data_source=
                step_process
                .properties
                .ProcessingOutputConfig
                .Outputs["train"]
                .S3Output
                .S3Uri,

            content_type="text/csv"
        )
    ]
)


step_train = TrainingStep(

    name="Training",

    step_args=train_args
)


# ============================================================
# 8. EVALUATION STEP
# ============================================================

evaluation_processor = ScriptProcessor(

    role=ROLE,

    image_uri=sklearn_image,

    command=["python3"],

    instance_type="ml.m5.large",

    instance_count=1,

    sagemaker_session=pipeline_session
)


evaluation_report = PropertyFile(

    name="EvaluationReport",

    output_name="evaluation",

    path="evaluation.json"
)


evaluation_args = evaluation_processor.run(

    code="evaluation.py",

    inputs=[

        # -------------------------
        # Model artifact
        # -------------------------

        ProcessingInput(

            input_name="model",

            s3_input=ProcessingS3Input(

                s3_uri=
                    step_train
                    .properties
                    .ModelArtifacts
                    .S3ModelArtifacts,

                s3_data_type="S3Prefix",

                local_path=
                    "/opt/ml/processing/model",

                s3_input_mode="File",

                s3_compression_type="None"
            )
        ),

        # -------------------------
        # Test dataset
        # -------------------------

        ProcessingInput(

            input_name="test",

            s3_input=ProcessingS3Input(

                s3_uri=
                    step_process
                    .properties
                    .ProcessingOutputConfig
                    .Outputs["test"]
                    .S3Output
                    .S3Uri,

                s3_data_type="S3Prefix",

                local_path=
                    "/opt/ml/processing/test",

                s3_input_mode="File",

                s3_compression_type="None"
            )
        )
    ],

    outputs=[

        ProcessingOutput(

            output_name="evaluation",

            s3_output=ProcessingS3Output(

                s3_uri=
                    f"s3://{BUCKET}/metrics/pipeline",

                local_path=
                    "/opt/ml/processing/evaluation",

                s3_upload_mode="EndOfJob"
            )
        )
    ],

    wait=False
)


step_evaluation = ProcessingStep(

    name="Evaluation",

    step_args=evaluation_args,

    property_files=[
        evaluation_report
    ]
)


# ============================================================
# 9. MODEL REGISTRATION
# ============================================================

print("\nPreparing Model Registration...")


model_builder = ModelBuilder(

    s3_model_data_url=
        step_train
        .properties
        .ModelArtifacts
        .S3ModelArtifacts,

    image_uri=sklearn_image,

    role_arn=ROLE,

    sagemaker_session=pipeline_session
)


# IMPORTANT:
# ModelStep requires ModelBuilder.register()
# in SageMaker SDK v3.

register_args = model_builder.register(

    model_package_group_name=MODEL_GROUP,

    content_types=[
        "text/csv"
    ],

    response_types=[
        "text/csv"
    ],

    inference_instances=[
        "ml.m5.large"
    ],

    transform_instances=[
        "ml.m5.large"
    ],

    approval_status=
        "PendingManualApproval",

    description=(
        "FreshMart Customer Membership "
        "Random Forest candidate model"
    )
)


step_register = ModelStep(

    name="ModelRegistration",

    step_args=register_args
)


# ============================================================
# 10. CREATE PIPELINE
# ============================================================

pipeline = Pipeline(

    name=PIPELINE_NAME,

    steps=[

        step_process,

        step_train,

        step_evaluation,

        step_register
    ],

    sagemaker_session=pipeline_session
)


# ============================================================
# 11. UPSERT PIPELINE
# ============================================================

print("\nCreating/updating SageMaker Pipeline...")


pipeline.upsert(
    role_arn=ROLE
)


print("\n" + "=" * 60)

print("PIPELINE CREATED SUCCESSFULLY")

print("=" * 60)

print(
    "Pipeline name:",
    PIPELINE_NAME
)


# ============================================================
# 12. START PIPELINE EXECUTION
# ============================================================

print("\nStarting pipeline execution...")


execution = pipeline.start()


print(
    "\nExecution ARN:"
)

print(
    execution.arn
)


# ============================================================
# 13. WAIT FOR EXECUTION
# ============================================================

print(
    "\nWaiting for pipeline execution..."
)

print(
    "This may take several minutes."
)


execution.wait()


# ============================================================
# 14. FINAL STATUS
# ============================================================

details = execution.describe()


status = details[
    "PipelineExecutionStatus"
]


print("\n" + "=" * 60)

print("FINAL PIPELINE STATUS")

print("=" * 60)

print(
    "Status:",
    status
)


if status == "Succeeded":

    print(
        "\nSUCCESS!"
    )

    print(
        "Task 4 completed successfully."
    )

    print(
        "\nPipeline:"
    )

    print(
        "Preprocessing -> Training -> "
        "Evaluation -> Model Registration"
    )

else:

    print(
        "\nPipeline did not succeed."
    )

    print(
        "Open SageMaker > Pipelines and "
        "check the failed step."
    )

FRESHMART SAGEMAKER MLOPS PIPELINE
Region       : us-east-1
Bucket       : bkt-deloitte-aug26-hyd
Input        : s3://bkt-deloitte-aug26-hyd/raw-data/FreshMart_Customer_Membership.csv
Target       : Premium_Member
Candidate    : Random Forest
Pipeline     : FreshMart-ML-Pipeline
Model Group  : FreshMart-Customer-Membership
Role         : arn:aws:iam::781843541123:role/service-role/AmazonSageMaker-ExecutionRole-20260827T100868

SageMaker SKLearn image:
683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-cpu-py3


/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


[08/27/26 08:58:13] INFO     StoppingCondition not provided. Using default:                         ]8;id=318637;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=318638;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

[08/27/26 08:58:14] INFO     OutputDataConfig not provided. Using default:                          ]8;id=318643;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=318644;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#153\153]8;;\
                             s3_output_path='s3://sagemaker-us-east-1-781843541123/freshmart-random                
                             -forest' kms_key_id=None compression_type='GZIP'                                      

                    INFO     Training image URI:                                               ]8;id=318649;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=318650;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-lea                     
                             rn:1.4-2-cpu-py3                                                                      

...

[08/27/26 08:58:15] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=318678;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=318679;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=318684;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=318685;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'CertifyForMarketplace' from the pipeline definition     ]8;id=318692;file:///opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py\model_step.py]8;;\:]8;id=318693;file:///opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py#195\195]8;;\
                             since it will be overridden in pipeline execution time.                               

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=318698;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=318699;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   


PIPELINE CREATED SUCCESSFULLY
Pipeline name: FreshMart-ML-Pipeline

Starting pipeline execution...

Execution ARN:
arn:aws:sagemaker:us-east-1:781843541123:pipeline/FreshMart-ML-Pipeline/execution/z4bh7t9v9m4d

Waiting for pipeline execution...
This may take several minutes.

FINAL PIPELINE STATUS
Status: Succeeded

SUCCESS!
Task 4 completed successfully.

Pipeline:
Preprocessing -> Training -> Evaluation -> Model Registration


# Task 5:

In [58]:
import boto3
from botocore.exceptions import ClientError

sm = boto3.client(
    "sagemaker",
    region_name=boto3.Session().region_name
)

MODEL_GROUP = "FreshMartMembershipModels"

# Check whether group exists
try:
    response = sm.describe_model_package_group(
        ModelPackageGroupName=MODEL_GROUP
    )

    print("✅ Model group already exists")
    print("Name:", response["ModelPackageGroupName"])
    print("ARN:", response["ModelPackageGroupArn"])

except ClientError as e:

    error_code = e.response["Error"]["Code"]

    if error_code in [
        "ValidationException",
        "ResourceNotFound"
    ]:

        print("Model group does not exist.")
        print("Creating:", MODEL_GROUP)

        response = sm.create_model_package_group(

            ModelPackageGroupName=MODEL_GROUP,

            ModelPackageGroupDescription=(
                "FreshMart Customer Membership "
                "Random Forest candidate models"
            )
        )

        print("\n✅ Model group created successfully")
        print(
            "ARN:",
            response["ModelPackageGroupArn"]
        )

    else:
        raise

✅ Model group already exists
Name: FreshMartMembershipModels
ARN: arn:aws:sagemaker:us-east-1:781843541123:model-package-group/FreshMartMembershipModels


In [60]:
import boto3
import json
from botocore.exceptions import ClientError

sm = boto3.client(
    "sagemaker",
    region_name=boto3.Session().region_name
)

BUCKET = "bkt-deloitte-aug26-hyd"

OLD_MODEL_GROUP = "FreshMart-Customer-Membership"

MODEL_GROUP = "FreshMartMembershipModels"

EVALUATION_S3_URI = (
    f"s3://{BUCKET}/metrics/pipeline/evaluation.json"
)


# ============================================================
# 1. GET CANDIDATE MODEL FROM TASK 4
# ============================================================

response = sm.list_model_packages(

    ModelPackageGroupName=OLD_MODEL_GROUP,

    SortBy="CreationTime",

    SortOrder="Descending",

    MaxResults=10
)

packages = response.get(
    "ModelPackageSummaryList",
    []
)

if not packages:
    raise Exception(
        f"No model found in {OLD_MODEL_GROUP}"
    )

candidate_arn = packages[0]["ModelPackageArn"]

candidate = sm.describe_model_package(
    ModelPackageName=candidate_arn
)

print("Candidate model:")
print(candidate_arn)


# ============================================================
# 2. GET MODEL ARTIFACT + IMAGE
# ============================================================

inference_spec = candidate[
    "InferenceSpecification"
]

container = inference_spec[
    "Containers"
][0]

MODEL_ARTIFACT = container[
    "ModelDataUrl"
]

IMAGE_URI = container[
    "Image"
]

content_types = inference_spec.get(
    "SupportedContentTypes",
    ["text/csv"]
)

response_types = inference_spec.get(
    "SupportedResponseMIMETypes",
    ["text/csv"]
)

print("\nModel artifact:")
print(MODEL_ARTIFACT)


# ============================================================
# 3. VERIFY MODEL GROUP
# ============================================================

try:

    group_details = sm.describe_model_package_group(

        ModelPackageGroupName=MODEL_GROUP
    )

    print(
        "\n✅ Model group exists:",
        MODEL_GROUP
    )

except ClientError as e:

    raise Exception(
        f"Model group {MODEL_GROUP} could not be found: {e}"
    )


# ============================================================
# 4. REGISTER MODEL
# ============================================================
#
# IMPORTANT:
# There is NO Tags parameter here.
#
# Tags belong to the Model Package Group,
# not to the Model Package Version.
#
# ============================================================

register_response = sm.create_model_package(

    ModelPackageGroupName=MODEL_GROUP,

    ModelPackageDescription=(
        "FreshMart Random Forest candidate model"
    ),

    ModelApprovalStatus="Approved",

    InferenceSpecification={

        "Containers": [

            {
                "Image": IMAGE_URI,

                "ModelDataUrl": MODEL_ARTIFACT
            }

        ],

        "SupportedContentTypes":
            content_types,

        "SupportedResponseMIMETypes":
            response_types,

        "SupportedRealtimeInferenceInstanceTypes": [

            "ml.m5.large"

        ],

        "SupportedTransformInstanceTypes": [

            "ml.m5.large"

        ]
    },

    # --------------------------------------------------------
    # Evaluation metrics
    # --------------------------------------------------------

    ModelMetrics={

        "ModelQuality": {

            "Statistics": {

                "ContentType":
                    "application/json",

                "S3Uri":
                    EVALUATION_S3_URI
            }
        }
    },

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    CustomerMetadataProperties={

        "Project":
            "FreshMart",

        "Algorithm":
            "RandomForest",

        "Dataset":
            "FreshMart_Customer_Membership.csv",

        "Split":
            "80:20",

        "RandomState":
            "42",

        "Evaluation":
            "Accuracy_Precision_Recall_F1score",

        "Pipeline":
            "FreshMart-ML-Pipeline"
    }
)


NEW_MODEL_ARN = (
    register_response[
        "ModelPackageArn"
    ]
)


print("\n" + "=" * 70)

print("✅ MODEL REGISTERED SUCCESSFULLY")

print("=" * 70)

print("\nModel Group:")
print(MODEL_GROUP)

print("\nModel Package ARN:")
print(NEW_MODEL_ARN)


# ============================================================
# 5. SET APPROVAL TO APPROVED
# ============================================================

sm.update_model_package(

    ModelPackageArn=NEW_MODEL_ARN,

    ModelApprovalStatus="Approved",

    ApprovalDescription=(
        "FreshMart Random Forest model approved "
        "after evaluation."
    )
)


print(
    "\n✅ Approval status set to Approved."
)


# ============================================================
# 6. VERIFY EVERYTHING
# ============================================================

final_model = sm.describe_model_package(

    ModelPackageName=NEW_MODEL_ARN
)


print("\n" + "=" * 70)

print("TASK 5 VERIFICATION")

print("=" * 70)


print(
    "\nModel Group:"
)

print(
    final_model[
        "ModelPackageGroupName"
    ]
)


print(
    "\nModel Version:"
)

print(
    final_model[
        "ModelPackageVersion"
    ]
)


print(
    "\nApproval Status:"
)

print(
    final_model[
        "ModelApprovalStatus"
    ]
)


print(
    "\nModel Artifact:"
)

print(
    final_model[
        "InferenceSpecification"
    ][
        "Containers"
    ][0][
        "ModelDataUrl"
    ]
)


print(
    "\nEvaluation Metrics:"
)

print(
    json.dumps(
        final_model.get(
            "ModelMetrics",
            {}
        ),
        indent=4
    )
)


# ============================================================
# 7. FINAL VALIDATION
# ============================================================

group_ok = (
    final_model[
        "ModelPackageGroupName"
    ]
    == MODEL_GROUP
)

approval_ok = (
    final_model[
        "ModelApprovalStatus"
    ]
    == "Approved"
)

artifact_ok = bool(
    final_model[
        "InferenceSpecification"
    ][
        "Containers"
    ][0].get(
        "ModelDataUrl"
    )
)

metrics_ok = bool(
    final_model.get(
        "ModelMetrics"
    )
)


print("\n" + "=" * 70)

if (
    group_ok
    and approval_ok
    and artifact_ok
    and metrics_ok
):

    print(
        "✅ TASK 5 COMPLETED SUCCESSFULLY"
    )

    print(
        "\nModel Group :",
        MODEL_GROUP
    )

    print(
        "Version     :",
        final_model[
            "ModelPackageVersion"
        ]
    )

    print(
        "Approval    : Approved"
    )

    print(
        "Artifact    : Present"
    )

    print(
        "Metrics     : Attached"
    )

else:

    print(
        "❌ TASK 5 VERIFICATION FAILED"
    )

print("=" * 70)

Candidate model:
arn:aws:sagemaker:us-east-1:781843541123:model-package/FreshMart-Customer-Membership/1

Model artifact:
s3://sagemaker-us-east-1-781843541123/freshmart-random-forest/pipelines-z4bh7t9v9m4d-Training-GLpTEnEGNS/output/model.tar.gz

✅ Model group exists: FreshMartMembershipModels

✅ MODEL REGISTERED SUCCESSFULLY

Model Group:
FreshMartMembershipModels

Model Package ARN:
arn:aws:sagemaker:us-east-1:781843541123:model-package/FreshMartMembershipModels/1

✅ Approval status set to Approved.

TASK 5 VERIFICATION

Model Group:
FreshMartMembershipModels

Model Version:
1

Approval Status:
Approved

Model Artifact:
s3://sagemaker-us-east-1-781843541123/freshmart-random-forest/pipelines-z4bh7t9v9m4d-Training-GLpTEnEGNS/output/model.tar.gz

Evaluation Metrics:
{
    "ModelQuality": {
        "Statistics": {
            "ContentType": "application/json",
            "S3Uri": "s3://bkt-deloitte-aug26-hyd/metrics/pipeline/evaluation.json"
        }
    }
}

✅ TASK 5 COMPLETED SUCCESS

# Task 6:

In [63]:
# ============================================================
# TASK 6: BATCH PREDICTION OUTPUT
# FINAL SIMPLE VERSION
# ============================================================

import pandas as pd
import boto3
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

BUCKET = "bkt-deloitte-aug26-hyd"

INPUT_FILE = "FreshMart_Customer_Membership.csv"

S3_INPUT_KEY = "raw-data/FreshMart_Customer_Membership.csv"

S3_OUTPUT_KEY = "predictions/customer_predictions.csv"


# ============================================================
# 1. DOWNLOAD ORIGINAL CSV FROM S3
# ============================================================

s3 = boto3.client("s3")

s3.download_file(
    BUCKET,
    S3_INPUT_KEY,
    INPUT_FILE
)

df = pd.read_csv(INPUT_FILE)

print("Original dataset shape:", df.shape)


# ============================================================
# 2. CLEAN DATA THE SAME WAY AS TASK 2
# ============================================================

# Remove duplicate customer records
df = df.drop_duplicates(
    subset=["Customer_ID"]
)

# Handle missing values
numeric_columns = df.select_dtypes(
    include=["number"]
).columns

categorical_columns = df.select_dtypes(
    include=["object"]
).columns

for col in numeric_columns:

    if df[col].isnull().any():

        df[col] = df[col].fillna(
            df[col].median()
        )


for col in categorical_columns:

    if df[col].isnull().any():

        df[col] = df[col].fillna(
            df[col].mode()[0]
        )


print(
    "Cleaned dataset shape:",
    df.shape
)


# ============================================================
# 3. CREATE SAME 80:20 SPLIT
# ============================================================

# Target
y = df["Premium_Member"]

# Features
X = df.drop(
    columns=[
        "Premium_Member",
        "Customer_ID",
        "Customer_Name"
    ]
)

# IMPORTANT:
# Keep Customer_ID separately
customer_ids = df["Customer_ID"]


# Same split used in Task 2
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(

    X,
    y,
    customer_ids,

    test_size=0.20,

    random_state=42
)


print(
    "\nTrain records:",
    len(X_train)
)

print(
    "Test records:",
    len(X_test)
)


# ============================================================
# 4. GENERATE RANDOM FOREST PREDICTIONS
# ============================================================

# 'rf' should be your Random Forest pipeline from Task 3

predictions = rf.predict(X_test)


print(
    "\nPredictions generated:",
    len(predictions)
)


# ============================================================
# 5. CREATE PREDICTION OUTPUT
# ============================================================

customer_predictions = pd.DataFrame({

    "Customer_ID":
        id_test.values,

    "Actual_Premium_Member":
        y_test.values,

    "Predicted_Premium_Member":
        predictions

})


# ============================================================
# 6. SELECT AT LEAST 20 RECORDS
# ============================================================

customer_predictions = (
    customer_predictions
    .head(20)
    .reset_index(drop=True)
)


# ============================================================
# 7. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 70)

print("CUSTOMER PREDICTIONS")

print("=" * 70)

print(
    customer_predictions.to_string(
        index=False
    )
)


# ============================================================
# 8. SAVE CSV
# ============================================================

local_file = "customer_predictions.csv"

customer_predictions.to_csv(
    local_file,
    index=False
)


print(
    "\n✅ CSV created:",
    local_file
)


# ============================================================
# 9. UPLOAD TO S3
# ============================================================

s3.upload_file(

    local_file,

    BUCKET,

    S3_OUTPUT_KEY
)


print(
    "\n✅ Uploaded successfully!"
)

print(
    "S3 Location:"
)

print(
    f"s3://{BUCKET}/{S3_OUTPUT_KEY}"
)


# ============================================================
# 10. VERIFY S3 FILE
# ============================================================

s3.head_object(

    Bucket=BUCKET,

    Key=S3_OUTPUT_KEY
)


# ============================================================
# FINAL RESULT
# ============================================================

print("\n" + "=" * 70)

print("✅ TASK 6 COMPLETED SUCCESSFULLY")

print("=" * 70)

print(
    "\nFile:",
    "customer_predictions.csv"
)

print(
    "Records:",
    len(customer_predictions)
)

print(
    "Columns:",
    list(customer_predictions.columns)
)

print(
    "\nS3:",
    f"s3://{BUCKET}/{S3_OUTPUT_KEY}"
)

Original dataset shape: (50500, 11)
Cleaned dataset shape: (50000, 11)

Train records: 40000
Test records: 10000

Predictions generated: 10000

CUSTOMER PREDICTIONS
Customer_ID  Actual_Premium_Member  Predicted_Premium_Member
    C033554                      0                         0
    C009428                      0                         0
    C000200                      0                         0
    C012448                      1                         0
    C039490                      0                         0
    C042725                      0                         0
    C010823                      0                         0
    C049499                      0                         0
    C004145                      0                         0
    C036959                      1                         0
    C043107                      1                         1
    C038696                      0                         0
    C006189                      0        